# training ANPEs on adroit

In [1]:
import os, sys 
import numpy as np

import torch
from torch import nn 
from torch.utils.tensorboard.writer import SummaryWriter

from sbi import utils as Ut
from sbi import inference as Inference

from sedflow import data as D
from sedflow import util as U

In [2]:
cuda = torch.cuda.is_available()
device = ("cuda:0" if cuda else "cpu")

seed = 12387
torch.manual_seed(seed)
if cuda:
    torch.cuda.manual_seed(seed)

In [3]:
bands = 'ugrizJ'
freez = False

In [4]:
D.load_modela?

In [5]:
x_train, y_train = D.load_modela('train', bands=bands, infer_redshift=freez)
print('Ntrain + Nvalid = %i' % (x_train.shape[0]))

Ntrain + Nvalid = 1000000


In [6]:
x_train = x_train[:10000]
y_train = y_train[:10000]

In [7]:
prior_low   = [7, 0., 0., 0., 0., 1e-2, np.log10(4.5e-5), np.log10(4.5e-5), 0, 0., -2.]
prior_high  = [12.5, 1., 1., 1., 1., 13.27, np.log10(1.5e-2), np.log10(1.5e-2), 3., 3., 1.]

lower_bounds = torch.tensor(prior_low).to(device)
upper_bounds = torch.tensor(prior_high).to(device)

prior = Ut.BoxUniform(low=lower_bounds, high=upper_bounds, device=device)

/home/chhahn/.conda/envs/torch-env/lib/python3.7/site-packages/sbi/utils/torchutils.py:28: UserWarning: GPU was selected as a device for training the neural network. Note that we expect **no** significant speed ups in training for the default architectures we provide. Using the GPU will be effective only for large neural networks with operations that are fast on the GPU, e.g., for a CNN or RNN `embedding_net`.
  "GPU was selected as a device for training the neural network. "


In [8]:
neural_posterior = Ut.posterior_nn('maf', 
        hidden_features=500, 
        num_transforms=10,   
        use_batch_norm=True)

In [9]:
anpe = Inference.SNPE(prior=prior,
        density_estimator=neural_posterior,
        device=device)

In [10]:
anpe.append_simulations(
    torch.as_tensor(x_train.astype(np.float32)).to(device),
    torch.as_tensor(y_train.astype(np.float32)).to(device))

In [11]:
p_x_y_est = anpe.train()

 Neural network successfully converged after 46 epochs.

In [12]:
anpe._summary['best_validation_log_probs']

{'median_observation_distances': [],
 'epochs': [46],
 'best_validation_log_probs': [-6.531898498535156],
 'validation_log_probs': [-12.72282763671875,
  -9.992079193115234,
  -9.312169036865235,
  -8.753645477294922,
  -8.867570770263672,
  -8.792292053222656,
  -8.069760284423829,
  -7.929667053222656,
  -7.8074219665527345,
  -7.817739349365234,
  -7.576341369628906,
  -7.934437622070313,
  -7.692656616210938,
  -7.389749755859375,
  -7.140921844482422,
  -7.223899291992187,
  -6.981431121826172,
  -6.946381072998047,
  -7.013062713623047,
  -6.814143737792969,
  -6.755107513427735,
  -6.930744384765625,
  -6.893671722412109,
  -6.716178283691407,
  -6.654632354736328,
  -6.531898498535156,
  -6.631691223144531,
  -6.600297973632813,
  -6.966599945068359,
  -6.885161437988281,
  -6.782747100830078,
  -6.847225433349609,
  -6.847997680664062,
  -6.796166748046875,
  -6.858311828613282,
  -6.696916625976563,
  -7.461062316894531,
  -7.261105895996094,
  -7.285428894042969,
  -7.216861